# 3.2 - Optimization - Multi-MIT and Hybrid Factor Analysis

Este notebook analisa o impacto do Intervalo Mínimo de Tempo (**MIT**: 30 min, 6h, 24h) e compara o comportamento dos 4 fatores isolados frente à **Propriedade Híbrida**.

In [ ]:
%config Completer.use_jedi = False
%load_ext autoreload
%autoreload 2

import sys, os, warnings
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')

PROJECT_ROOT = Path.cwd() if Path.cwd().name == 'src' else Path.cwd() / 'src'
sys.path.insert(0, str(PROJECT_ROOT.parent))

from src.config import ROOT, MITS_MINUTES
from src.utils.logger import save_result, save_figure

RESULTS_DIR = ROOT / 'src' / '5 - Results'
FIGURES_DIR = ROOT / 'src' / '4 - Figures'

## 1. Carregamento dos Resultados de QC

In [ ]:
df_qc = pd.read_parquet(RESULTS_DIR / 'qc_classification_results.parquet')
print('Distribuição por MIT:')
print(df_qc.groupby('mit_minutes')['hybrid_quality'].value_counts())

## 2. Comparativo entre Fatores Isolados e Propriedade Híbrida

In [ ]:
factors = ['yearly_rainfall_quality', 'rainfall_event_quality', 'rainfall_intensity_quality', 'rainfall_duration_quality', 'hybrid_quality']
summary_by_mit = []

for mit_val in df_qc['mit_minutes'].unique():
    subset = df_qc[df_qc['mit_minutes'] == mit_val]
    for factor in factors:
        hq_pct = (subset[factor] == 'HQ').mean() * 100
        summary_by_mit.append({
            'mit_minutes': mit_val,
            'factor': factor,
            'hq_percentage': round(hq_pct, 2)
        })

df_comp = pd.DataFrame(summary_by_mit)
df_pivot = df_comp.pivot(index='factor', columns='mit_minutes', values='hq_percentage')
print(df_pivot)

## 3. Visualização Gráfica do Desempenho por MIT

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
df_pivot.plot(kind='bar', ax=ax)
ax.set_ylabel('% Estações Classificadas como HQ')
ax.set_title('Sensibilidade dos Fatores de Qualidade por MIT')
ax.grid(axis='y', linestyle='--', alpha=0.7)
plt.xticks(rotation=30, ha='right')
save_figure(fig, 'factors_comparison_by_mit.png', subdir='4 - Figures')
plt.show()

save_result(df_comp, 'mit_factors_summary.parquet', subdir='5 - Results')